# Multi-Session LeJEPA Experiment Runner

Train and evaluate the LeJEPA teacher model on multi-session neural spike data.

**Order of operations:**
1. Set `CONFIG_PATH` in the Config cell.
2. Run all cells top to bottom.
3. Results (CSV + plots) are saved under `results_out_path` from your config.
4. Training curves, test metrics, and latent-space UMAP also render inline below.

**Prerequisite:** Run the dataset pipeline first (`experiments/data/create_dataset.py`) so a Parquet exists at your config's `data_path`.

Clone Repo

In [ ]:
!git clone https://github.com/jacobposchl/jepsyn
!pip install snntorch temporaldata torch_brain umap-learn pyarrow pyyaml

In [ ]:
import os
import sys
from pathlib import Path

# The repo is cloned to /content/jepsyn in Colab.
REPO_ROOT = Path("/content/jepsyn")
os.chdir(REPO_ROOT)
print(f"CWD: {Path.cwd()}")

# jepsyn/ package is importable directly from repo root.
# Add experiments/multi_session/ so multi_session.py can be imported by name.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "multi_session"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from multi_session import evaluate_model, identify_units, load_and_prepare_data, save_results, train_lejepa
from jepsyn.utils import verify_config

print("Imports OK")

## Configuration

Point `CONFIG_PATH` at your experiment YAML. See `configs/lejepa_lif_visual_cortex.yaml` for a full template with all encoder, predictor, and training fields.

At minimum, `results_out_path` must be set in the config for checkpoints and plots to be saved to disk.

In [ ]:
# Path is relative to repo root (where you ran %cd jepsyn)
CONFIG_PATH = Path("experiments/multi_session/configs/lejepa_lif_visual_cortex.yaml")

config = verify_config(CONFIG_PATH)

print(f"Config loaded:      {CONFIG_PATH}")
print(f"data_path:          {config['data_path']}")
print(f"results_out_path:   {config.get('results_out_path', '(not set — results will not be saved to disk)')}")
print()
tcfg = config["training_config"]
mcfg = config["model_config"]
print(f"epochs:             {tcfg.get('epochs', 100)}")
print(f"batch_size:         {tcfg.get('batch_size', 32)}")
print(f"lr:                 {tcfg.get('lr', 1e-4)}")
print(f"mask_ratio:         {tcfg.get('mask_ratio', 0.5)}")
print(f"ema_momentum:       {tcfg.get('ema_momentum', 0.996)}")
print(f"lambd (SIGReg):     {tcfg.get('lambd', 0.05)}")
print()
print(f"d_model:            {mcfg.get('d_model', 256)}")
print(f"n_latents:          {mcfg.get('n_latents', 64)}")
print(f"max_time_ms:        {mcfg.get('max_time_ms', 400)}")

## Data Loading

Loads the Parquet from `data_path`, validates schema and integrity, then splits windows by `session_id` into train / val / test sets.

In [ ]:
train_loader, val_loader, test_loader, unit_maps, test_session_ids = load_and_prepare_data(config)

print(f"\nSessions in unit_maps: {len(unit_maps)}")
print(f"Units per session:     min={min(len(m) for m in unit_maps.values())}, "
      f"max={max(len(m) for m in unit_maps.values())}")
print(f"Train batches:         {len(train_loader)}")
print(f"Val batches:           {len(val_loader)}")
print(f"Test batches:          {len(test_loader)}")
print(f"Test session IDs:      {test_session_ids}")

## LeJEPA Training

Trains the **context encoder** (online, gradients flow), **target encoder** (EMA copy, no gradients), and **predictor** (narrow Transformer).

- Loss: `(1 - λ) * MSE(h_pred, h_tgt) + λ * SIGReg(h_ctx, h_tgt)`
- After training, the checkpoint is saved to `<results_out_path>/lejepa_checkpoint.pt`.

In [ ]:
!git pull
import importlib
import jepsyn.losses.lejepa
import jepsyn.losses
importlib.reload(jepsyn.losses.lejepa)
importlib.reload(jepsyn.losses)


In [ ]:
jepa_model, jepa_train_metrics = train_lejepa(config, train_loader, val_loader, unit_maps)
print("\nTraining complete.")
jepa_train_metrics.tail()

## Training Results

Saves `metrics.csv` and `training_curves.png` to `<results_out_path>/LeJEPA/training/`, then renders the curves inline.

In [ ]:
save_results(stage="LeJEPA", phase="training", metrics=jepa_train_metrics, config=config)

In [ ]:
# Inline training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("LeJEPA - Training Curves")

axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_loss"], label="train")
axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Total Loss")
axes[0].legend()

axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_pred_loss"], label="pred loss")
axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_reg_loss"], label="reg loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Pred vs Reg Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Full per-epoch metrics table
jepa_train_metrics

## LeJEPA Test Evaluation

Runs the trained model on the held-out test set (no masking — context encoder sees all events).

Metrics per batch:
- **`pred_loss`**: MSE between predicted and target mean-pooled representations `[B, D]`
- **`cos_similarity`**: cosine similarity between context and target representations

## Unit Identification

Adapts the test-session unit embedding tables using the self-supervised LeJEPA objective, with all shared weights frozen.

The test sessions' unit embeddings were randomly initialized during training and never updated (no test data was seen during training). Unit identification fixes this by running the JEPA prediction loss on test windows for a small number of steps — only the unit embedding parameters for test sessions are trainable. This takes ~1 minute on GPU.

After this step the test representations are meaningful rather than driven by random unit-ID fingerprints.

In [ ]:
identify_units(jepa_model, test_loader, test_session_ids, config)

In [ ]:
jepa_test_metrics = evaluate_model(jepa_model, test_loader, stage="LeJEPA")

In [ ]:
save_results(stage="LeJEPA", phase="test", metrics=jepa_test_metrics, config=config)

# Display saved plots if results_out_path is set
results_path = config.get("results_out_path")
if results_path:
    test_bar_path = Path(results_path) / "LeJEPA" / "test" / "test_metrics.png"
    latent_path   = Path(results_path) / "LeJEPA" / "test" / "latent_space.png"
    if test_bar_path.exists():
        print("Test metrics bar chart:")
        display(Image(filename=str(test_bar_path)))
    if latent_path.exists():
        print("Latent space (UMAP):")
        display(Image(filename=str(latent_path)))

## Download Results

In [ ]:
!zip -r LeJEPA_export.zip results
from google.colab import files
files.download('results.zip')